In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option('display.max_columns', None)
sns.set(style="whitegrid")

# Load data with the same header fix as before
df = pd.read_csv("default of credit card clients.csv", skiprows=1)

# Rename target and drop ID (same as EDA notebook)
df.rename(columns={"default payment next month": "default"}, inplace=True)
df.drop(columns=["ID"], inplace=True)

df.shape

(30000, 24)

In [3]:
# Check duplicates before removing
print("Duplicates before:", df.duplicated().sum())

# Remove duplicate rows
df = df.drop_duplicates()

# Confirm removal
print("Duplicates after:", df.duplicated().sum())
print("New shape:", df.shape)

Duplicates before: 35
Duplicates after: 0
New shape: (29965, 24)


In [4]:
# EDUCATION should only be 1=graduate school, 2=university, 3=high school, 4=others
# Values 0, 5, 6 are undocumented -> group them into 4 (others)
print("Before:\n", df['EDUCATION'].value_counts())

df['EDUCATION'] = df['EDUCATION'].replace({0: 4, 5: 4, 6: 4})

print("\nAfter:\n", df['EDUCATION'].value_counts())

Before:
 EDUCATION
2    14019
1    10563
3     4915
5      280
4      123
6       51
0       14
Name: count, dtype: int64

After:
 EDUCATION
2    14019
1    10563
3     4915
4      468
Name: count, dtype: int64


In [5]:
# MARRIAGE should only be 1=married, 2=single, 3=others
# Value 0 is undocumented -> group into 3 (others)
print("Before:\n", df['MARRIAGE'].value_counts())

df['MARRIAGE'] = df['MARRIAGE'].replace({0: 3})

print("\nAfter:\n", df['MARRIAGE'].value_counts())

Before:
 MARRIAGE
2    15945
1    13643
3      323
0       54
Name: count, dtype: int64

After:
 MARRIAGE
2    15945
1    13643
3      377
Name: count, dtype: int64


In [6]:
# Function to detect outlier bounds using IQR
def get_outlier_bounds(column):
    Q1 = df[column].quantile(0.25)
    Q3 = df[column].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    return lower, upper

# Check outlier counts for key numeric columns
outlier_cols = ['LIMIT_BAL', 'BILL_AMT1', 'PAY_AMT1', 'AGE']

for col in outlier_cols:
    lower, upper = get_outlier_bounds(col)
    outliers = df[(df[col] < lower) | (df[col] > upper)]
    print(f"{col}: {len(outliers)} outliers (bounds: {lower:.1f} to {upper:.1f})")

LIMIT_BAL: 167 outliers (bounds: -235000.0 to 525000.0)
BILL_AMT1: 2386 outliers (bounds: -91902.5 to 162757.5)
PAY_AMT1: 2742 outliers (bounds: -5012.0 to 11020.0)
AGE: 272 outliers (bounds: 8.5 to 60.5)


In [7]:
# Cap outliers at the IQR bounds instead of removing rows (avoids losing too much data)
def cap_outliers(column):
    lower, upper = get_outlier_bounds(column)
    df[column] = np.where(df[column] < lower, lower, df[column])
    df[column] = np.where(df[column] > upper, upper, df[column])

for col in outlier_cols:
    cap_outliers(col)

# Confirm - recheck outlier counts (should be 0 now)
for col in outlier_cols:
    lower, upper = get_outlier_bounds(col)
    outliers = df[(df[col] < lower) | (df[col] > upper)]
    print(f"{col}: {len(outliers)} outliers remaining")

LIMIT_BAL: 0 outliers remaining
BILL_AMT1: 0 outliers remaining
PAY_AMT1: 0 outliers remaining
AGE: 0 outliers remaining


In [8]:
# Split into features (X) and target (y)
X = df.drop(columns=['default'])
y = df['default']

print("Features shape:", X.shape)
print("Target shape:", y.shape)

Features shape: (29965, 23)
Target shape: (29965,)


In [10]:
from sklearn.model_selection import train_test_split

# 80% train, 20% test, stratify to preserve class balance in both sets
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print("Train shape:", X_train.shape)
print("Test shape:", X_test.shape)

Train shape: (23972, 23)
Test shape: (5993, 23)


In [11]:
from sklearn.preprocessing import StandardScaler

# Standardize numeric features (mean=0, std=1)
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Convert back to DataFrame for readability
X_train_scaled = pd.DataFrame(X_train_scaled, columns=X_train.columns)
X_test_scaled = pd.DataFrame(X_test_scaled, columns=X_test.columns)

X_train_scaled.head()

,LIMIT_BAL,SEX,EDUCATION,MARRIAGE,AGE,PAY_0,PAY_2,PAY_3,PAY_4,PAY_5,PAY_6,BILL_AMT1,BILL_AMT2,BILL_AMT3,BILL_AMT4,BILL_AMT5,BILL_AMT6,PAY_AMT1,PAY_AMT2,PAY_AMT3,PAY_AMT4,PAY_AMT5,PAY_AMT6
0,1.041092,-1.22974,0.215889,-1.070453,-0.047010,-1.771947,-1.566494,-1.544266,-1.530037,-1.538794,-1.492534,-0.865841,-0.694464,-0.678531,-0.674057,-0.665085,-0.652938,-1.001051,-0.245685,-0.294973,-0.320892,-0.318684,-0.289319
1,-0.911787,0.81318,1.561518,2.760389,0.173072,1.793738,1.781057,1.818359,1.906811,2.010259,2.005012,0.062270,-0.005582,0.032115,0.072324,-0.220084,-0.208968,-0.343362,-0.183578,-0.294973,-0.093938,-0.318684,-0.223376
2,-0.677441,-1.22974,-1.129740,0.844968,-1.367506,-0.880526,-0.729606,-0.703609,-0.670825,0.235732,0.256239,-0.857197,-0.688208,-0.672134,-0.660258,-0.650460,-0.645486,-0.874089,-0.227301,-0.244835,-0.291254,-0.318684,-0.232278
3,-0.911787,0.81318,0.215889,0.844968,-1.037382,-0.880526,-0.729606,-0.703609,-0.670825,-0.651531,-0.618147,-0.858249,-0.665187,-0.672912,-0.667997,-0.665085,-0.639846,-0.406843,-0.229537,-0.272953,-0.320892,-0.266810,-0.289319
4,0.259940,-1.22974,0.215889,0.844968,-0.597217,-1.771947,-1.566494,-1.544266,-1.530037,-1.538794,-1.492534,0.078155,0.009043,-0.321904,1.244053,1.513376,1.516122,2.150138,0.784923,6.772372,0.868346,1.428331,-0.050330


In [12]:
# Save the cleaned dataset (duplicates removed, categories fixed, outliers capped)
# This is the dataset before scaling/splitting - useful as your "cleaned data" submission
df.to_csv("cleaned_credit_default.csv", index=False)

print("Cleaned dataset saved as 'cleaned_credit_default.csv'")
print("Final shape:", df.shape)

Cleaned dataset saved as 'cleaned_credit_default.csv'
Final shape: (29965, 24)


In [13]:
# Save train/test splits so Section 5 (Model Building) can load them directly
X_train_scaled.to_csv("X_train.csv", index=False)
X_test_scaled.to_csv("X_test.csv", index=False)
y_train.to_csv("y_train.csv", index=False)
y_test.to_csv("y_test.csv", index=False)

print("Train/test files saved successfully.")

Train/test files saved successfully.
